In [ ]:
%%capture
!pip install pyvi transformers transformers[torch] evaluate
!git clone --single-branch --branch fast_tokenizers_BARTpho_PhoBERT_BERTweet https://github.com/datquocnguyen/transformers.git
%cd transformers
!pip install -e .

In [ ]:
import os
import evaluate
import torch
import collections
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
from pyvi import ViTokenizer
from datasets import Dataset
from huggingface_hub import login
import wandb
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments, pipeline

login('your_huggingface_auth_token_here')
os.environ["WANDB_DISABLED"] = "true"
metric = evaluate.load("squad")

In [ ]:
num_batch = 2
learning_rate = 2e-5
num_epochs = 5

base_checkpoint = "vinai/phobert-base-v2"
custom_checkpoint = "./models/ViLegalBERT"

In [ ]:
df = pd.DataFrame(r"./datasets/ALQAC/EQA/eqa.csv")
df

In [ ]:
df['context'] = df['context'].apply(ViTokenizer.tokenize)
df['question'] = df['question'].apply(ViTokenizer.tokenize)
df['answer'] = df['answer'].apply(ViTokenizer.tokenize)

In [ ]:
def find_answer(question, answer):
    start_idx = question.find(answer)
    return start_idx

df['answer_start'] = df.apply(lambda row: find_answer(row['context'], row['answer']), axis=1)
df = df[df['answer_start'] != -1]
df

In [ ]:
dataset = Dataset.from_pandas(df, preserve_index=False)

temp = []
for i in dataset:
    res = {}
    res['context'] = i['context']
    res['question'] = i['question']
    res['answers'] = {'text': [i['answer']], 'answer_start': [i['answer_start']]}
    temp.append(res)

df = pd.DataFrame(temp)
df

In [ ]:
total_rows = len(df)

each_partition_rows = int(0.2 * total_rows)

partition_1 = df.sample(n=each_partition_rows, random_state=42)
df = df.drop(partition_1.index)

partition_2 = df.sample(n=each_partition_rows, random_state=42)
df = df.drop(partition_2.index)

partition_3 = df.sample(n=each_partition_rows, random_state=42)
df = df.drop(partition_3.index)

partition_4 = df.sample(n=each_partition_rows, random_state=42)
df = df.drop(partition_4.index)

partition_5 = df

print("Partition 1:", len(partition_1))
print("Partition 2:", len(partition_2))
print("Partition 3:", len(partition_3))
print("Partition 4:", len(partition_4))
print("Partition 5:", len(partition_5))

In [ ]:
k_fold = [partition_1, partition_2, partition_3, partition_4, partition_5]

In [ ]:
max_length = 256
stride = 50

def preprocess_training_examples(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        max_length= max_length,
        truncation="only_second",
        stride= stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []
    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [ ]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["question"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [ ]:
def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(examples):
        example_id = example["question"]
        context = example["context"]
        answers = []

        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["question"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [ ]:
n_best = 50
max_answer_length = 512
predicted_answers = []

res = []

em_results = []
f1_results = []

for i in range(len(k_fold)):
    
    tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
    model = AutoModelForQuestionAnswering.from_pretrained(custom_checkpoint)
    
    test = k_fold[i]
    train = pd.concat(k_fold[:i]+ k_fold[i+1:])
    train_set = Dataset.from_pandas(train, preserve_index=False)
    test_set = Dataset.from_pandas(test, preserve_index=False)
    train_dataset = train_set.map(preprocess_training_examples, batched=True, remove_columns=train_set.column_names)
    test_dataset = test_set.map(preprocess_validation_examples, batched=True, remove_columns=test_set.column_names)
    
    args = TrainingArguments(
        "bert-finetuned-squad",
        evaluation_strategy="no",
        save_strategy = "no",
        eval_steps=500,
        save_strategy="epoch",
        save_total_limit = 1,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        per_device_train_batch_size=num_batch,
        per_device_eval_batch_size=num_batch,
        learning_rate= learning_rate,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        fp16=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        tokenizer=tokenizer,
    )

    print("==========TRAINING==========")
    trainer.train()

    print("==========SAVING FINE-TUNED MODEL==========")
    trainer.save_model(f'./working/model_{i+1}')

    print("==========EVALUATING MODEL PERFORMANCE==========")
    predictions, _, _ = trainer.predict(test_dataset)
    start_logits, end_logits = predictions
    x = compute_metrics(start_logits, end_logits, test_dataset, test_set)
    res.append(x)
    print(f"result_model_{i+1}", x)

In [ ]:
avg_exact_match = sum(d['exact_match'] for d in res) / len(res)
avg_f1 = sum(d['f1'] for d in res) / len(res)

print(f"Trung bình exact_match: {avg_exact_match}")
print(f"Trung bình f1: {avg_f1}")